# ANARCI + MMseqs2 TCR CDR Analysis

This Colab notebook provides a clean workflow for:

1. Installing and validating **ANARCI** and **MMseqs2**.
2. Reading a FASTA containing all chains from TCR–peptide–MHC complexes.
3. Identifying TCR alpha (`A`) and beta (`B`) variable domains with ANARCI.
4. Extracting **CDR1-IMGT, CDR2-IMGT, and CDR3-IMGT** without losing insertion-coded residues.
5. Logging short, invalid, and non-TCR sequences instead of silently discarding them.
6. Exporting separate alpha/beta full-chain, variable-domain, CDR3, and junction FASTAs.
7. Clustering and searching the variable-domain FASTAs with MMseqs2.

Official references: [ANARCI](https://pypi.org/project/ANARCI/),
[IMGT CDR definitions](https://www.imgt.org/IMGTScientificChart/Nomenclature/IMGT-FRCDRdefinition.html),
and [MMseqs2](https://github.com/soedinglab/MMseqs2).


## 0. Start with a fresh Colab runtime

Use **Runtime → Disconnect and delete runtime**, reconnect, and then run this notebook from the top.

This notebook intentionally installs only the ANARCI implementation used by the `anarci()` API below.
Do **not** install `anarci-mhc`, clone `oxpig/ANARCI`, copy files into `~/.anarci`, or run the original
project's `requirements.txt` in this runtime. Those installation paths were the source of the mixed
package/data-model problem in the earlier notebook.


In [ ]:
# Install HMMER, which ANARCI uses for variable-domain recognition.
!apt-get -qq update
!apt-get -qq install -y hmmer

# %pip targets the Python environment used by the current Colab kernel.
%pip install -q "biopython==1.85" "ANARCI==2026.2.13.2"


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libdivsufsort3:amd64.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../libdivsufsort3_2.0.1-5_amd64.deb ...
Unpacking libdivsufsort3:amd64 (2.0.1-5) ...
Selecting previously unselected package hmmer.
Preparing to unpack .../hmmer_3.3.2+dfsg-1_amd64.deb ...
Unpacking hmmer (3.3.2+dfsg-1) ...
Setting up libdivsufsort3:amd64 (2.0.1-5) ...
Setting up hmmer (3.3.2+dfsg-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.13) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.r

In [ ]:
%%bash
set -euo pipefail

cd /content

# Use the fastest official static MMseqs2 build supported by the Colab CPU.
if grep -qm1 avx2 /proc/cpuinfo; then
  mmseqs_build="avx2"
else
  mmseqs_build="sse41"
fi

wget -q -O mmseqs-linux.tar.gz \
  "https://mmseqs.com/latest/mmseqs-linux-${mmseqs_build}.tar.gz"
tar -xzf mmseqs-linux.tar.gz
ln -sf /content/mmseqs/bin/mmseqs /usr/local/bin/mmseqs

mmseqs version


17b688d21dda57fc5f5b7286ecba7ec003d4717f


## 1. Validate the installation

All checks below must pass before analyzing the full FASTA. The HMM database should be inside the
installed ANARCI package; no manual model-building or copying step is needed.


In [ ]:
import importlib.metadata
import json
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import anarci
import pandas as pd
from anarci.anarci import HMM_path

hmmscan_path = shutil.which("hmmscan")
mmseqs_path = shutil.which("mmseqs")
hmm_database = Path(HMM_path) / "ALL.hmm"
mmseqs_version = subprocess.check_output(
    ["mmseqs", "version"], text=True
).strip()

print("Python:", sys.version.split()[0])
print("ANARCI:", anarci.__version__)
print("hmmscan:", hmmscan_path)
print("ANARCI HMM database:", hmm_database)
print("HMM database exists:", hmm_database.is_file())
print("MMseqs2:", mmseqs_version)

assert hmmscan_path is not None, "hmmscan was not found"
assert hmm_database.is_file(), "ANARCI's ALL.hmm database was not found"
assert mmseqs_path is not None, "MMseqs2 was not found"

OUTPUT_DIR = Path("/content/ANARCI_MMseqs2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

run_metadata = {
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "anarci_version": anarci.__version__,
    "biopython_version": importlib.metadata.version("biopython"),
    "pandas_version": pd.__version__,
    "mmseqs2_version": mmseqs_version,
    "numbering_scheme": "IMGT",
}


Python: 3.12.13
ANARCI: 2026.2.13.2
hmmscan: /usr/bin/hmmscan
ANARCI HMM database: /usr/local/lib/python3.12/dist-packages/anarci/dat/HMMs/ALL.hmm
HMM database exists: True
MMseqs2: 17b688d21dda57fc5f5b7286ecba7ec003d4717f


### ANARCI control test

The earlier notebook used a 30-residue fragment, which is too short for a reliable full variable-domain
test. This known full-length control should be recognized and numbered.


In [ ]:
from anarci import anarci

control_sequence = (
    "EVQLQQSGAEVVRSGASVKLSCTASGFNIKDYYIHWVKQRPEKGLEWIGWIDPEIGDTEYV"
    "PKFQGKATMTADTSSNTAYLQLSSLTSEDTAVYYCNAGHDYDRGRFPYWGQGTLVTVSA"
)

control_numbering, control_details, control_hits = anarci(
    [("ANARCI_control", control_sequence)],
    scheme="imgt",
    output=False,
)

assert control_numbering[0] is not None, "ANARCI did not number the control"

control_info = control_details[0][0]
control_domain = control_numbering[0][0][0]
print("Control chain type:", control_info["chain_type"])
print("Control species model:", control_info["species"])
print("Control bit score:", control_info["bitscore"])
print("Numbered residues:", sum(aa != "-" for _, aa in control_domain))
print("ANARCI control passed")


Control chain type: H
Control species model: mouse
Control bit score: 176.2
Numbered residues: 120
ANARCI control passed


## 2. Upload `all_sequences.fasta`

Upload the FASTA containing all chains. It is expected that MHC chains, beta-2 microglobulin, peptides,
and other non-TCR sequences will receive **no alpha/beta TCR hit**. Those are logged rather than treated
as installation failures.


In [ ]:
from google.colab import files

default_fasta = Path("/content/all_sequences.fasta")

if not default_fasta.exists():
    print("Upload all_sequences.fasta")
    uploaded = files.upload()
    uploaded_fastas = [
        Path("/content") / name
        for name in uploaded
        if name.lower().endswith((".fasta", ".fa", ".faa", ".fas"))
    ]
    if not uploaded_fastas:
        raise FileNotFoundError("No FASTA file was uploaded")
    FASTA_PATH = next(
        (path for path in uploaded_fastas if path.name == "all_sequences.fasta"),
        uploaded_fastas[0],
    )
else:
    FASTA_PATH = default_fasta

print("Using:", FASTA_PATH)


Upload all_sequences.fasta


Saving all_sequences.fasta to all_sequences.fasta
Using: /content/all_sequences.fasta


## 3. Inspect and validate the input sequences

ANARCI is only run on sequences at least 70 residues long and containing the 20 standard amino-acid
letters. Sequences containing `X` or another nonstandard symbol are preserved in the failure report for
manual review; they are not silently changed to a different amino acid.


In [ ]:
from Bio import SeqIO

VALID_AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWY")
records = list(SeqIO.parse(str(FASTA_PATH), "fasta"))

if not records:
    raise ValueError(f"No FASTA records were found in {FASTA_PATH}")

input_rows = []
candidate_rows = []
prefilter_failures = []

for source_index, record in enumerate(records):
    sequence = str(record.seq).upper()
    invalid_letters = sorted(set(sequence) - VALID_AMINO_ACIDS)
    analysis_id = f"seq{source_index:06d}"

    item = {
        "analysis_id": analysis_id,
        "source_index": source_index,
        "source_id": record.id,
        "description": record.description,
        "sequence": sequence,
        "input_length": len(sequence),
        "invalid_letters": "".join(invalid_letters),
    }
    input_rows.append(item)

    if len(sequence) < 70:
        prefilter_failures.append({
            **item,
            "failure_stage": "prefilter",
            "failure_reason": "sequence_shorter_than_70_aa",
        })
    elif invalid_letters:
        prefilter_failures.append({
            **item,
            "failure_stage": "prefilter",
            "failure_reason": "nonstandard_amino_acid_letter",
        })
    else:
        candidate_rows.append(item)

print("Total FASTA records:", len(records))
print("Candidates sent to ANARCI:", len(candidate_rows))
print("Short sequences logged:", sum(
    row["failure_reason"] == "sequence_shorter_than_70_aa"
    for row in prefilter_failures
))
print("Sequences with nonstandard letters logged:", sum(
    row["failure_reason"] == "nonstandard_amino_acid_letter"
    for row in prefilter_failures
))

display(pd.DataFrame(input_rows)[[
    "analysis_id", "source_id", "input_length", "invalid_letters", "description"
]].head(10))


Total FASTA records: 1861
Candidates sent to ANARCI: 1511
Short sequences logged: 348
Sequences with nonstandard letters logged: 2


,analysis_id,source_id,input_length,invalid_letters,description
0,seq000000,1AO7_2|Chain,100,,1AO7_2|Chain B|BETA-2 MICROGLOBULIN|Homo sapie...
1,seq000001,1AO7_3|Chain,9,,1AO7_3|Chain C|TAX PEPTIDE|Human T-lymphotropi...
2,seq000002,1AO7_4|Chain,204,,1AO7_4|Chain D|T CELL RECEPTOR ALPHA|Homo sapi...
3,seq000003,1AO7_5|Chain,245,,1AO7_5|Chain E|T CELL RECEPTOR BETA|Homo sapie...
4,seq000004,1AO7_1|Chain,275,,1AO7_1|Chain A|HLA-A 0201|Homo sapiens (9606)
5,seq000005,1BD2_1|Chain,275,,1BD2_1|Chain A|HLA-A 0201|Homo sapiens (9606)
6,seq000006,1BD2_2|Chain,100,,1BD2_2|Chain B|BETA-2 MICROGLOBULIN|Homo sapie...
7,seq000007,1BD2_5|Chain,244,,1BD2_5|Chain E|T CELL RECEPTOR BETA|Homo sapie...
8,seq000008,1BD2_4|Chain,204,,1BD2_4|Chain D|T CELL RECEPTOR ALPHA|Homo sapi...
9,seq000009,1BD2_3|Chain,9,,1BD2_3|Chain C|TAX PEPTIDE|Human T-lymphotropi...


## 4. Run ANARCI once on the candidate batch

Running the sequences together avoids starting a separate HMMER subprocess for every FASTA record.
`allow={"A", "B"}` keeps only TCR alpha and beta variable domains.


In [ ]:
if not candidate_rows:
    raise ValueError("No valid sequences remained after input filtering")

sequence_pairs = [
    (row["analysis_id"], row["sequence"])
    for row in candidate_rows
]

numbered_results, detail_results, hit_tables = anarci(
    sequence_pairs,
    scheme="imgt",
    allow={"A", "B"},
    output=False,
    ncpu=2,
)

assert len(numbered_results) == len(candidate_rows)
assert len(detail_results) == len(candidate_rows)
print("ANARCI finished for", len(candidate_rows), "candidate sequences")


Limiting hmmer search to species ['human', 'mouse'] was requested but hits did not achieve a high enough bitscore. Reverting to using any species
ANARCI finished for 1511 candidate sequences


## 5. Extract IMGT regions without dropping insertions

ANARCI returns positions as `(integer_position, insertion_code)` pairs. A long CDR3 may contain entries
such as `111A`, `111B`, or `112A`. Converting the numbering to a dictionary keyed only by the integer
position would overwrite those residues. The functions below preserve ANARCI's original ordered list.

IMGT definitions used here:

| Region | IMGT positions | Notes |
|---|---:|---|
| CDR1-IMGT | 27–38 | V-gene encoded |
| CDR2-IMGT | 56–65 | V-gene encoded |
| CDR3-IMGT | 105–117 | Excludes conserved Cys104 and Phe/Trp118 |
| Junction | 104–118 | Includes those conserved flanking residues |


In [ ]:
IMGT_REGIONS = {
    "FR1_IMGT": (1, 26),
    "CDR1_IMGT": (27, 38),
    "FR2_IMGT": (39, 55),
    "CDR2_IMGT": (56, 65),
    "FR3_IMGT": (66, 104),
    "CDR3_IMGT": (105, 117),
    # This ANARCI release numbers IMGT positions through 128.
    "FR4_IMGT": (118, 128),
    "JUNCTION_IMGT": (104, 118),
}


def region_entries(numbered_domain, start, end):
    # Return ordered, occupied ANARCI positions in an inclusive IMGT range.
    return [
        ((position, str(insertion).strip()), aa)
        for ((position, insertion), aa) in numbered_domain
        if start <= position <= end and aa != "-"
    ]


def region_sequence(numbered_domain, start, end):
    return "".join(
        aa for _, aa in region_entries(numbered_domain, start, end)
    )


def region_numbering_string(numbered_domain, start, end):
    return " ".join(
        f"{position}{insertion}:{aa}"
        for ((position, insertion), aa) in region_entries(
            numbered_domain, start, end
        )
    )


def occupied_sequence(numbered_domain):
    # Reconstruct the complete numbered variable domain, including insertions.
    return "".join(aa for _, aa in numbered_domain if aa != "-")


# Unit check: insertion-coded residues must remain in their original order.
synthetic_numbering = [
    ((105, ""), "A"),
    ((111, ""), "B"),
    ((111, "A"), "C"),
    ((112, "A"), "D"),
    ((112, ""), "E"),
    ((117, ""), "F"),
]
assert region_sequence(synthetic_numbering, 105, 117) == "ABCDEF"
print("Insertion-preserving extraction check passed")


Insertion-preserving extraction check passed


In [ ]:
result_rows = []
failure_rows = list(prefilter_failures)

for item, numbered_entry, details_entry in zip(
    candidate_rows, numbered_results, detail_results
):
    if numbered_entry is None or details_entry is None:
        failure_rows.append({
            **item,
            "failure_stage": "ANARCI",
            "failure_reason": "no_alpha_or_beta_TCR_hit",
        })
        continue

    for domain_index, (domain, info) in enumerate(
        zip(numbered_entry, details_entry)
    ):
        numbered_domain, seq_start, seq_end = domain
        chain_type = info["chain_type"]

        if chain_type not in {"A", "B"}:
            failure_rows.append({
                **item,
                "failure_stage": "ANARCI",
                "failure_reason": f"unexpected_chain_type_{chain_type}",
            })
            continue

        row = {
            "analysis_id": item["analysis_id"],
            "source_index": item["source_index"],
            "source_id": item["source_id"],
            "description": item["description"],
            "domain_index": domain_index,
            "chain_type": chain_type,
            "species_model": info.get("species"),
            "evalue": info.get("evalue"),
            "bitscore": info.get("bitscore"),
            "seqstart_index": seq_start,
            "seqend_index": seq_end,
            "input_length": item["input_length"],
            "raw_input_sequence": item["sequence"],
            "variable_domain_sequence": occupied_sequence(numbered_domain),
        }

        for region_name, (start, end) in IMGT_REGIONS.items():
            sequence = region_sequence(numbered_domain, start, end)
            row[region_name] = sequence
            row[f"{region_name}_length"] = len(sequence)

        row["CDR1_IMGT_numbered"] = region_numbering_string(
            numbered_domain, 27, 38
        )
        row["CDR2_IMGT_numbered"] = region_numbering_string(
            numbered_domain, 56, 65
        )
        row["CDR3_IMGT_numbered"] = region_numbering_string(
            numbered_domain, 105, 117
        )
        row["CDR3_has_insertion_code"] = any(
            105 <= position <= 117 and bool(str(insertion).strip())
            for ((position, insertion), aa) in numbered_domain
            if aa != "-"
        )
        result_rows.append(row)

cdr_df = pd.DataFrame(result_rows)
failure_df = pd.DataFrame(failure_rows)

if cdr_df.empty:
    raise RuntimeError(
        "ANARCI found no alpha/beta TCR variable domains. "
        "Review the input headers, sequences, and failure report."
    )

cdr_results_path = OUTPUT_DIR / "TCR_CDR_IMGT_results.csv"
failure_path = OUTPUT_DIR / "ANARCI_sequence_failures.csv"
cdr_df.to_csv(cdr_results_path, index=False)
failure_df.to_csv(failure_path, index=False)

print("TCR domains saved:", len(cdr_df))
print(cdr_df["chain_type"].value_counts().rename(index={"A": "Alpha", "B": "Beta"}))
print("CDR3s containing insertion codes:", int(cdr_df["CDR3_has_insertion_code"].sum()))
print("Logged non-results:", len(failure_df))
print("Saved:", cdr_results_path)
print("Saved:", failure_path)


TCR domains saved: 732
chain_type
Beta     378
Alpha    354
Name: count, dtype: int64
CDR3s containing insertion codes: 92
Logged non-results: 1130
Saved: /content/ANARCI_MMseqs2_outputs/TCR_CDR_IMGT_results.csv
Saved: /content/ANARCI_MMseqs2_outputs/ANARCI_sequence_failures.csv


## 6. Validate and inspect the CDR output

`CDR3_IMGT` normally does **not** begin with the conserved cysteine or end with the conserved
phenylalanine/tryptophan. Use `JUNCTION_IMGT` when you want positions 104–118 including those flanks.
The numbered columns are included so individual boundaries can be audited directly.


In [ ]:
summary_columns = [
    "source_id",
    "chain_type",
    "bitscore",
    "CDR1_IMGT",
    "CDR1_IMGT_length",
    "CDR2_IMGT",
    "CDR2_IMGT_length",
    "CDR3_IMGT",
    "CDR3_IMGT_length",
    "JUNCTION_IMGT",
    "CDR3_has_insertion_code",
]
display(cdr_df[summary_columns].head(10))

print("CDR length summary by chain type")
display(
    cdr_df.groupby("chain_type")[[
        "CDR1_IMGT_length", "CDR2_IMGT_length", "CDR3_IMGT_length"
    ]].describe().round(2)
)

# Show the detailed numbering for the first TCR domain.
audit_row = cdr_df.iloc[0]
print("Audit sequence:", audit_row["source_id"], "chain", audit_row["chain_type"])
print("CDR1:", audit_row["CDR1_IMGT_numbered"])
print("CDR2:", audit_row["CDR2_IMGT_numbered"])
print("CDR3:", audit_row["CDR3_IMGT_numbered"])


,source_id,chain_type,bitscore,CDR1_IMGT,CDR1_IMGT_length,CDR2_IMGT,CDR2_IMGT_length,CDR3_IMGT,CDR3_IMGT_length,JUNCTION_IMGT,CDR3_has_insertion_code
0,1AO7_4|Chain,A,129.9,DRGSQS,6,IYSNGD,6,AVTTDSWGKLQ,11,CAVTTDSWGKLQF,False
1,1AO7_5|Chain,B,143.1,MNHEY,5,SVGAGI,6,ASRPGLAGGRPEQY,14,CASRPGLAGGRPEQYF,True
2,1BD2_5|Chain,B,148.9,MNHEY,5,SVGAGI,6,ASSYPGGGFYEQY,13,CASSYPGGGFYEQYF,False
3,1FO0_5|Chain,B,106.3,NSQYPW,6,LRSPGD,6,TCSADRVGNTLY,12,CTCSADRVGNTLYF,False
4,1G6R_2|Chains,B,135.1,NNHNN,5,SYGAGS,6,ASGGGGTLY,9,CASGGGGTLYF,False
5,1G6R_1|Chains,A,135.6,YSATPY,6,YYSGDPVV,8,AVSGFASALT,10,CAVSGFASALTF,False
6,1KJ2_5|Chains,B,107.3,NSQYPW,6,LRSPGD,6,TCSAAPDWGASAETLY,16,CTCSAAPDWGASAETLYF,True
7,1KJ2_4|Chains,A,147.8,DSTFNY,6,IRSVSDK,7,AARYQGGRALI,11,CAARYQGGRALIF,False
8,1LP9_5|Chains,B,142.2,NNHDY,5,SYVADS,6,ASSDWVSYEQY,11,CASSDWVSYEQYF,False
9,1LP9_4|Chains,A,127.9,STYSPF,6,SFTDNKR,7,ALFLASSSFSKLV,13,CALFLASSSFSKLVF,False


CDR length summary by chain type


CDR1_IMGT_length                                       \
                      count  mean   std  min  25%  50%  75%  max   
chain_type                                                         
A                     354.0  6.06  0.59  5.0  6.0  6.0  6.0  8.0   
B                     378.0  5.08  0.28  5.0  5.0  5.0  5.0  6.0   

           CDR2_IMGT_length        ...           CDR3_IMGT_length         \
                      count  mean  ...  75%  max            count   mean   
chain_type                         ...                                     
A                     354.0  6.48  ...  7.0  8.0            354.0  11.33   
B                     378.0  6.01  ...  6.0  7.0            378.0  11.91   

                                               
             std  min   25%   50%   75%   max  
chain_type                                     
A           1.63  6.0  10.0  11.0  13.0  16.0  
B           1.58  9.0  11.0  12.0  13.0  18.0  

[2 rows x 24 columns]

Audit sequence: 1AO7_4|Chain chain A
CDR1: 27:D 28:R 29:G 36:S 37:Q 38:S
CDR2: 56:I 57:Y 58:S 63:N 64:G 65:D
CDR3: 105:A 106:V 107:T 108:T 109:D 110:S 113:W 114:G 115:K 116:L 117:Q


## 7. Export alpha and beta FASTAs

Four sequence types are exported separately for each chain:

- **Full input:** the original FASTA sequence, potentially including constant domains.
- **Variable domain:** only residues numbered by ANARCI; best for general TCR-variable similarity.
- **CDR3-IMGT:** positions 105–117 without Cys104/Phe-or-Trp118.
- **Junction:** positions 104–118 including the conserved flanking residues.


In [ ]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord


def export_fasta(frame, sequence_column, output_path):
    output_records = []
    mapping_rows = []

    for row in frame.itertuples(index=False):
        sequence = getattr(row, sequence_column)
        if not isinstance(sequence, str) or not sequence:
            continue

        output_id = (
            f"{row.analysis_id}_D{row.domain_index}_{row.chain_type}"
        )
        output_records.append(
            SeqRecord(
                Seq(sequence),
                id=output_id,
                description=(
                    f"source_id={row.source_id} "
                    f"chain_type={row.chain_type} "
                    f"sequence_type={sequence_column}"
                ),
            )
        )
        mapping_rows.append({
            "output_id": output_id,
            "analysis_id": row.analysis_id,
            "source_id": row.source_id,
            "description": row.description,
            "domain_index": row.domain_index,
            "chain_type": row.chain_type,
            "sequence_type": sequence_column,
        })

    count = SeqIO.write(output_records, str(output_path), "fasta")
    return count, mapping_rows


fasta_outputs = {}
all_mapping_rows = []

sequence_types = {
    "full_input": "raw_input_sequence",
    "variable_domains": "variable_domain_sequence",
    "CDR3_IMGT": "CDR3_IMGT",
    "junction_104_118": "JUNCTION_IMGT",
}

for chain_type, chain_name in {"A": "alpha", "B": "beta"}.items():
    subset = cdr_df[cdr_df["chain_type"] == chain_type].copy()

    for file_label, sequence_column in sequence_types.items():
        output_path = OUTPUT_DIR / f"TCR_{chain_name}_{file_label}.fasta"
        count, mapping_rows = export_fasta(
            subset, sequence_column, output_path
        )
        fasta_outputs[(chain_type, file_label)] = output_path
        all_mapping_rows.extend(mapping_rows)
        print(output_path.name, "sequences:", count)

sequence_map_df = pd.DataFrame(all_mapping_rows)
sequence_map_path = OUTPUT_DIR / "TCR_exported_sequence_id_map.csv"
sequence_map_df.to_csv(sequence_map_path, index=False)
print("Saved:", sequence_map_path)


TCR_alpha_full_input.fasta sequences: 354
TCR_alpha_variable_domains.fasta sequences: 354
TCR_alpha_CDR3_IMGT.fasta sequences: 354
TCR_alpha_junction_104_118.fasta sequences: 354
TCR_beta_full_input.fasta sequences: 378
TCR_beta_variable_domains.fasta sequences: 378
TCR_beta_CDR3_IMGT.fasta sequences: 378
TCR_beta_junction_104_118.fasta sequences: 378
Saved: /content/ANARCI_MMseqs2_outputs/TCR_exported_sequence_id_map.csv


## 8. CDR3 transition counts and probabilities

The original notebook attempted to save `prob_matrix` before creating it. This section calculates
separate alpha and beta transition matrices. Each probability is conditioned on the amino acid in the row.


In [ ]:
import numpy as np

AMINO_ACID_ORDER = list("ACDEFGHIKLMNPQRSTVWY")


def transition_matrices(sequences):
    counts = pd.DataFrame(
        0,
        index=AMINO_ACID_ORDER,
        columns=AMINO_ACID_ORDER,
        dtype=int,
    )

    for sequence in sequences.dropna().astype(str):
        for aa1, aa2 in zip(sequence, sequence[1:]):
            if aa1 in counts.index and aa2 in counts.columns:
                counts.loc[aa1, aa2] += 1

    row_totals = counts.sum(axis=1).replace(0, np.nan)
    probabilities = counts.div(row_totals, axis=0).fillna(0.0)
    return counts, probabilities


for chain_type, chain_name in {"A": "alpha", "B": "beta"}.items():
    chain_sequences = cdr_df.loc[
        cdr_df["chain_type"] == chain_type, "CDR3_IMGT"
    ]
    counts, probabilities = transition_matrices(chain_sequences)

    counts_path = OUTPUT_DIR / f"TCR_{chain_name}_CDR3_transition_counts.csv"
    probabilities_path = (
        OUTPUT_DIR / f"TCR_{chain_name}_CDR3_transition_probabilities.csv"
    )
    counts.to_csv(counts_path)
    probabilities.to_csv(probabilities_path)
    print(chain_name, "transitions:", int(counts.to_numpy().sum()))


alpha transitions: 3656
beta transitions: 4124


## 9. Cluster alpha and beta variable domains with MMseqs2

ANARCI and MMseqs2 are consecutive stages, not one combined program:

**all chains → ANARCI classification/numbering → alpha and beta variable-domain FASTAs → MMseqs2**

The default clustering thresholds below require at least 70% sequence identity and 80% coverage.
Alpha and beta chains are clustered separately. Adjust the parameters if your scientific question requires
a stricter or more permissive definition of similarity.


In [ ]:
import uuid

MMSEQS_MIN_SEQ_ID = 0.70
MMSEQS_COVERAGE = 0.80
MMSEQS_COV_MODE = 1
MMSEQS_THREADS = 2

mmseqs_run_dir = OUTPUT_DIR / "mmseqs2" / uuid.uuid4().hex[:10]
mmseqs_run_dir.mkdir(parents=True, exist_ok=True)


def fasta_record_count(path):
    return sum(1 for _ in SeqIO.parse(str(path), "fasta"))


def run_easy_cluster(input_fasta, label):
    input_fasta = Path(input_fasta)
    sequence_count = fasta_record_count(input_fasta)
    if sequence_count < 2:
        print(f"Skipping {label}: only {sequence_count} sequence(s)")
        return None

    result_prefix = mmseqs_run_dir / f"{label}_clusters"
    temporary_dir = mmseqs_run_dir / f"{label}_cluster_tmp"

    command = [
        "mmseqs",
        "easy-cluster",
        str(input_fasta),
        str(result_prefix),
        str(temporary_dir),
        "--min-seq-id", str(MMSEQS_MIN_SEQ_ID),
        "-c", str(MMSEQS_COVERAGE),
        "--cov-mode", str(MMSEQS_COV_MODE),
        "--threads", str(MMSEQS_THREADS),
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)
    return result_prefix


cluster_prefixes = {}
for chain_type, chain_name in {"A": "alpha", "B": "beta"}.items():
    input_fasta = fasta_outputs[(chain_type, "variable_domains")]
    cluster_prefixes[chain_type] = run_easy_cluster(
        input_fasta, f"TCR_{chain_name}_variable"
    )

run_metadata.update({
    "mmseqs_cluster_min_seq_id": MMSEQS_MIN_SEQ_ID,
    "mmseqs_cluster_coverage": MMSEQS_COVERAGE,
    "mmseqs_cluster_cov_mode": MMSEQS_COV_MODE,
    "mmseqs_threads": MMSEQS_THREADS,
})


Running: mmseqs easy-cluster /content/ANARCI_MMseqs2_outputs/TCR_alpha_variable_domains.fasta /content/ANARCI_MMseqs2_outputs/mmseqs2/61c04876c7/TCR_alpha_variable_clusters /content/ANARCI_MMseqs2_outputs/mmseqs2/61c04876c7/TCR_alpha_variable_cluster_tmp --min-seq-id 0.7 -c 0.8 --cov-mode 1 --threads 2
Running: mmseqs easy-cluster /content/ANARCI_MMseqs2_outputs/TCR_beta_variable_domains.fasta /content/ANARCI_MMseqs2_outputs/mmseqs2/61c04876c7/TCR_beta_variable_clusters /content/ANARCI_MMseqs2_outputs/mmseqs2/61c04876c7/TCR_beta_variable_cluster_tmp --min-seq-id 0.7 -c 0.8 --cov-mode 1 --threads 2


In [ ]:
cluster_member_tables = {}

for chain_type, chain_name in {"A": "alpha", "B": "beta"}.items():
    prefix = cluster_prefixes.get(chain_type)
    if prefix is None:
        continue

    cluster_tsv = Path(f"{prefix}_cluster.tsv")
    members = pd.read_csv(
        cluster_tsv,
        sep="\t",
        names=["representative", "member"],
    )
    members["chain_type"] = chain_type
    members["cluster_size"] = members.groupby(
        "representative"
    )["member"].transform("size")
    members = members.sort_values(
        ["cluster_size", "representative", "member"],
        ascending=[False, True, True],
    )

    output_csv = OUTPUT_DIR / f"MMseqs2_{chain_name}_cluster_members.csv"
    members.to_csv(output_csv, index=False)
    cluster_member_tables[chain_type] = members

    print(
        chain_name,
        "sequences:", len(members),
        "clusters:", members["representative"].nunique(),
    )
    display(members.head(10))


alpha sequences: 354 clusters: 37


,representative,member,chain_type,cluster_size
239,seq000860_D0_A,seq000002_D0_A,A,61
240,seq000860_D0_A,seq000052_D0_A,A,61
241,seq000860_D0_A,seq000056_D0_A,A,61
242,seq000860_D0_A,seq000061_D0_A,A,61
243,seq000860_D0_A,seq000105_D0_A,A,61
244,seq000860_D0_A,seq000176_D0_A,A,61
245,seq000860_D0_A,seq000179_D0_A,A,61
246,seq000860_D0_A,seq000208_D0_A,A,61
247,seq000860_D0_A,seq000212_D0_A,A,61
248,seq000860_D0_A,seq000259_D0_A,A,61


beta sequences: 378 clusters: 32


,representative,member,chain_type,cluster_size
0,seq000003_D0_B,seq000003_D0_B,B,67
1,seq000003_D0_B,seq000007_D0_B,B,67
2,seq000003_D0_B,seq000053_D0_B,B,67
3,seq000003_D0_B,seq000055_D0_B,B,67
4,seq000003_D0_B,seq000060_D0_B,B,67
5,seq000003_D0_B,seq000065_D0_B,B,67
6,seq000003_D0_B,seq000070_D0_B,B,67
7,seq000003_D0_B,seq000075_D0_B,B,67
8,seq000003_D0_B,seq000098_D0_B,B,67
9,seq000003_D0_B,seq000099_D0_B,B,67


## 10. MMseqs2 all-vs-all searches without self-hits

This produces interpretable pairwise hit tables for the alpha and beta variable domains. The results are
local sequence-alignment hits that pass the MMseqs2 search threshold; they are not a guaranteed dense
matrix containing every possible pair.


In [ ]:
MMSEQS_SEARCH_EVALUE = 1e-3
MMSEQS_SEARCH_COVERAGE = 0.50

MMSEQS_COLUMNS = [
    "query",
    "target",
    "pident",
    "alnlen",
    "mismatch",
    "gapopen",
    "qstart",
    "qend",
    "tstart",
    "tend",
    "evalue",
    "bits",
    "qcov",
    "tcov",
]


def run_all_vs_all(input_fasta, label):
    input_fasta = Path(input_fasta)
    sequence_count = fasta_record_count(input_fasta)
    if sequence_count < 2:
        print(f"Skipping {label}: only {sequence_count} sequence(s)")
        return pd.DataFrame(columns=MMSEQS_COLUMNS)

    raw_tsv = mmseqs_run_dir / f"{label}_all_vs_all.tsv"
    temporary_dir = mmseqs_run_dir / f"{label}_search_tmp"
    format_output = ",".join(MMSEQS_COLUMNS)

    command = [
        "mmseqs",
        "easy-search",
        str(input_fasta),
        str(input_fasta),
        str(raw_tsv),
        str(temporary_dir),
        "-e", str(MMSEQS_SEARCH_EVALUE),
        "-c", str(MMSEQS_SEARCH_COVERAGE),
        "--cov-mode", "0",
        "--threads", str(MMSEQS_THREADS),
        "--format-output", format_output,
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

    if not raw_tsv.exists() or raw_tsv.stat().st_size == 0:
        return pd.DataFrame(columns=MMSEQS_COLUMNS)

    hits = pd.read_csv(raw_tsv, sep="\t", names=MMSEQS_COLUMNS)
    hits = hits[hits["query"] != hits["target"]].copy()
    return hits.sort_values(
        ["query", "evalue", "bits"],
        ascending=[True, True, False],
    )


search_tables = {}
for chain_type, chain_name in {"A": "alpha", "B": "beta"}.items():
    hits = run_all_vs_all(
        fasta_outputs[(chain_type, "variable_domains")],
        f"TCR_{chain_name}_variable",
    )
    search_tables[chain_type] = hits
    output_csv = OUTPUT_DIR / f"MMseqs2_{chain_name}_all_vs_all_no_self.csv"
    hits.to_csv(output_csv, index=False)
    print(chain_name, "non-self hits:", len(hits))
    display(hits.head(10))

run_metadata.update({
    "mmseqs_search_evalue": MMSEQS_SEARCH_EVALUE,
    "mmseqs_search_coverage": MMSEQS_SEARCH_COVERAGE,
})


Running: mmseqs easy-search /content/ANARCI_MMseqs2_outputs/TCR_alpha_variable_domains.fasta /content/ANARCI_MMseqs2_outputs/TCR_alpha_variable_domains.fasta /content/ANARCI_MMseqs2_outputs/mmseqs2/61c04876c7/TCR_alpha_variable_all_vs_all.tsv /content/ANARCI_MMseqs2_outputs/mmseqs2/61c04876c7/TCR_alpha_variable_search_tmp -e 0.001 -c 0.5 --cov-mode 0 --threads 2 --format-output query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,qcov,tcov
alpha non-self hits: 93195


,query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,qcov,tcov
1,seq000002_D0_A,seq000179_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0
2,seq000002_D0_A,seq000355_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0
3,seq000002_D0_A,seq000208_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0
4,seq000002_D0_A,seq000052_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0
5,seq000002_D0_A,seq000056_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0
6,seq000002_D0_A,seq000061_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0
7,seq000002_D0_A,seq000259_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0
8,seq000002_D0_A,seq000105_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0
9,seq000002_D0_A,seq000284_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0
10,seq000002_D0_A,seq001354_D0_A,100.0,110,0,0,1,110,1,110,4.307000e-73,230,1.0,1.0


Running: mmseqs easy-search /content/ANARCI_MMseqs2_outputs/TCR_beta_variable_domains.fasta /content/ANARCI_MMseqs2_outputs/TCR_beta_variable_domains.fasta /content/ANARCI_MMseqs2_outputs/mmseqs2/61c04876c7/TCR_beta_variable_all_vs_all.tsv /content/ANARCI_MMseqs2_outputs/mmseqs2/61c04876c7/TCR_beta_variable_search_tmp -e 0.001 -c 0.5 --cov-mode 0 --threads 2 --format-output query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,qcov,tcov
beta non-self hits: 111558


,query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,qcov,tcov
68766,seq000003_D0_B,seq000177_D0_B,100.0,115,0,0,1,115,1,115,3.632000e-78,245,1.000,1.0
68767,seq000003_D0_B,seq000178_D0_B,100.0,115,0,0,1,115,1,115,3.632000e-78,245,1.000,1.0
68768,seq000003_D0_B,seq000209_D0_B,100.0,115,0,0,1,115,1,115,3.632000e-78,245,1.000,1.0
68769,seq000003_D0_B,seq000260_D0_B,100.0,115,0,0,1,115,1,115,3.632000e-78,245,1.000,1.0
68770,seq000003_D0_B,seq000104_D0_B,100.0,115,0,0,1,115,1,115,3.632000e-78,245,1.000,1.0
68771,seq000003_D0_B,seq000285_D0_B,100.0,115,0,0,1,115,1,115,3.632000e-78,245,1.000,1.0
68772,seq000003_D0_B,seq000053_D0_B,100.0,113,0,0,3,115,1,113,8.557000e-77,241,0.983,1.0
68773,seq000003_D0_B,seq000055_D0_B,100.0,113,0,0,3,115,1,113,8.557000e-77,241,0.983,1.0
68774,seq000003_D0_B,seq000060_D0_B,100.0,113,0,0,3,115,1,113,8.557000e-77,241,0.983,1.0
68775,seq000003_D0_B,seq000354_D0_B,96.5,115,4,0,1,115,1,115,2.765000e-75,237,1.000,1.0


## 11. Exact CDR3 groups

CDR3 sequences are short, so full-length variable-domain clustering is usually the clearer MMseqs2
analysis. This table separately reports exact duplicate CDR3-IMGT sequences within alpha and beta chains.


In [ ]:
exact_cdr3_groups = (
    cdr_df.groupby(["chain_type", "CDR3_IMGT"], dropna=False)
    .agg(
        sequence_count=("source_id", "size"),
        source_ids=("source_id", lambda values: ";".join(map(str, values))),
    )
    .reset_index()
    .sort_values(
        ["chain_type", "sequence_count", "CDR3_IMGT"],
        ascending=[True, False, True],
    )
)
exact_cdr3_path = OUTPUT_DIR / "TCR_exact_CDR3_IMGT_groups.csv"
exact_cdr3_groups.to_csv(exact_cdr3_path, index=False)
display(exact_cdr3_groups.head(20))


,chain_type,CDR3_IMGT,sequence_count,source_ids
156,A,AVTTDSWGKLQ,12,1AO7_4|Chain;1QRN_4|Chain;1QSE_4|Chain;1QSF_4|...
76,A,AMRGDSSYKLI,9,3UTS_4|Chains;3UTT_4|Chains;5C07_4|Chains;5C08...
139,A,AVSAKGTGSKLS,9,3TF7_3|Chains;3TFK_3|Chain;3TJH_3|Chain;3TPU_1...
7,A,AALYGNEKIT,7,5M00_3|Chain;5M01_3|Chain;5M02_3|Chain;5TIL_6|...
114,A,AVNFGGGKLI,7,3QDG_4|Chain;3QDJ_4|Chain;4L3E_4|Chain;6AM5_4|...
42,A,AGAGSQGNLI,6,1OGA_4|Chain;2VLJ_4|Chain;2VLK_4|Chain;2VLR_4|...
122,A,AVQDLGTSGSRLT,6,3MV7_4|Chain;3MV8_4|Chain;3MV9_4|Chain;4PRH_4|...
113,A,AVNDGGRLT,5,4JFD_4|Chain;4JFE_4|Chain;4JFF_4|Chain;6EQB_4|...
116,A,AVNNARLM,5,9NMU_4|Chain;9NMV_4|Chain;9NMW_4|Chain;9NMX_4|...
119,A,AVNVAGKST,5,3HG1_4|Chain;4QOK_4|Chain;6EQA_4|Chain;7Q99_4|...


## 12. Save run metadata and download all results

The ZIP contains the CDR table, failure log, exported FASTAs, transition matrices, MMseqs2 cluster files,
MMseqs2 non-self search hits, and the exact CDR3 groups.


In [ ]:
run_metadata.update({
    "input_fasta": FASTA_PATH.name,
    "input_record_count": len(records),
    "anarci_candidate_count": len(candidate_rows),
    "tcr_domain_count": len(cdr_df),
    "alpha_domain_count": int((cdr_df["chain_type"] == "A").sum()),
    "beta_domain_count": int((cdr_df["chain_type"] == "B").sum()),
    "logged_failure_count": len(failure_df),
})

metadata_path = OUTPUT_DIR / "run_metadata.json"
metadata_path.write_text(
    json.dumps(run_metadata, indent=2), encoding="utf-8"
)

archive_base = "/content/ANARCI_MMseqs2_results"
archive_path = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=str(OUTPUT_DIR),
)

print("Results directory:", OUTPUT_DIR)
print("ZIP archive:", archive_path)
files.download(archive_path)


Results directory: /content/ANARCI_MMseqs2_outputs
ZIP archive: /content/ANARCI_MMseqs2_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretation checklist

- A large `no_alpha_or_beta_TCR_hit` count is expected when the input contains all five complex chains.
- Compare expected alpha/beta counts with the number of complexes, but investigate missing structures,
  duplicate entities, nonstandard residues, and incomplete variable domains before assuming a fixed 2× count.
- Use `CDR3_IMGT` for the strict IMGT region and `JUNCTION_IMGT` for the common Cys-to-Phe/Trp representation.
- Audit `CDR3_IMGT_numbered` whenever a sequence has an insertion code or an unexpected length.
- Use the variable-domain FASTAs for MMseqs2 chain-level comparisons; keep alpha and beta analyses separate.
